# 00_chunk_pdfs_semantic.ipynb — semantic chunk PDFs → JSONL (Ollama)

Reads all PDFs from `./pdf/` and writes one combined JSONL to `data/chunks.jsonl`.

Requires Ollama:
- chat model: `qwen3`
- embedding model: `all-minilm:l12-v2`

Optional `.env`:
- `OLLAMA_BASE_URL` (default `http://localhost:11434`)
- `OLLAMA_EMBED_MODEL` (default `all-minilm:l12-v2`)
- `SEMANTIC_THRESHOLD_TYPE` (default `percentile`)
- `SEMANTIC_THRESHOLD` (default `95`)
- `MIN_CHUNK_CHARS` (default `400`)


In [1]:
!pip install -U cryptography



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# %pip install -q python-dotenv tqdm pypdf langchain-experimental langchain-community langchain-ollama

import os, json, re
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm
from pypdf import PdfReader
from langchain_experimental.text_splitter import SemanticChunker

try:
    from langchain_ollama import OllamaEmbeddings
except Exception:
    from langchain_community.embeddings import OllamaEmbeddings

load_dotenv()

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
EMBEDDING_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "all-minilm:l12-v2")

PDF_DIR = Path("pdf")
OUT_PATH = Path("data/chunks.jsonl")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

THRESHOLD_TYPE = os.getenv("SEMANTIC_THRESHOLD_TYPE", "percentile")
THRESHOLD = float(os.getenv("SEMANTIC_THRESHOLD", "95"))
MIN_CHUNK_CHARS = int(os.getenv("MIN_CHUNK_CHARS", "400"))

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type=THRESHOLD_TYPE,
    breakpoint_threshold_amount=THRESHOLD,
)

def read_pdf_with_page_markers(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r"\s+\n", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        parts.append(f"\n\n[[PAGE:{i}]]\n\n{text}")
    return "".join(parts).strip()

def page_range_from_chunk_text(chunk_text: str):
    pages = [int(m) for m in re.findall(r"\[\[PAGE:(\d+)\]\]", chunk_text)]
    if not pages:
        return None, None
    return min(pages), max(pages)

def strip_page_markers(chunk_text: str) -> str:
    return re.sub(r"\s*\[\[PAGE:\d+\]\]\s*", "\n", chunk_text).strip()

pdfs = sorted(PDF_DIR.glob("*.pdf"))
assert pdfs, f"No PDFs found in {PDF_DIR.resolve()}"
print(f"Found {len(pdfs)} pdf(s).")


KeyboardInterrupt: 

In [ ]:
total = 0
with OUT_PATH.open("w", encoding="utf-8") as out:
    for pdf_path in tqdm(pdfs, desc="PDFs"):
        doc_id = pdf_path.name
        raw = read_pdf_with_page_markers(pdf_path)

        docs = chunker.create_documents([raw])
        from langchain_text_splitters import RecursiveCharacterTextSplitter

        # tune these
        CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1200"))        # chars
        CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "150"))   # chars

        sizer = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

        final_docs = []
        for d in docs:
            # split each semantic chunk into smaller overlapping pieces
            final_docs.extend(sizer.split_documents([d]))

        docs = final_docs

        for idx, d in enumerate(docs, start=1):
            txt = d.page_content
            p1, p2 = page_range_from_chunk_text(txt)
            clean = strip_page_markers(txt)

            if len(clean) < MIN_CHUNK_CHARS:
                continue

            chunk_id = f"{doc_id}#{idx:06d}"
            record = {
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "text": clean,
                "metadata": {
                    "source": str(pdf_path.as_posix()),
                    "page_start": p1,
                    "page_end": p2,
                },
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            total += 1

print(f"✅ wrote {total} chunks → {OUT_PATH.resolve()}")


PDFs:   0%|          | 0/1 [00:00<?, ?it/s]

✅ wrote 153 chunks → C:\Users\tarmo\Desktop\llmgenai\LLM-course-2025_Tarmo\week-5\graph_rag\data\chunks.jsonl
